# dev: save / load / export_h5ad

Checks for `pdata.save()`, `pAnnData.load()` and `pdata.export_h5ad()` on a PD import and a DIA-NN import.

For each import: save the fresh object → load → compare, then run a chain of analysis steps → save → load → compare,
then keep working on the **loaded** object to make sure it still behaves like the original.

`same(a, b)` uses the strict comparator from the test suite (`tests/conftest.py::assert_pdata_equal`): every AnnData slot,
`.rs`, `.summary`, `.stats` (incl. `df.attrs`), `.history`, `.metadata`, with types. It raises on the first mismatch.

Output files go to `dev/saveload_out/` (delete when done).

In [1]:
from scpviz import pAnnData as pAnnData
from scpviz import plotting as scplt
from scpviz import utils as scutils
import scanpy as sc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context='paper', style='ticks')

import os, sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'tests'))
from conftest import assert_pdata_equal

out = Path('saveload_out')
out.mkdir(exist_ok=True)
for f in out.iterdir():   # start clean so the overwrite checks behave the same on rerun
    f.unlink()

def same(a, b, history_tail=1):
    # history_tail=1: the loaded object has one extra 'Loaded from ...' history line
    assert_pdata_equal(a, b, history_tail=history_tail)
    print('✅ identical (apart from the last', history_tail, 'history line)')

def size(path):
    print(f'{path}: {os.path.getsize(path) / 1e6:.1f} MB')

# PD import (`pd_prot.txt` + `pd_pep.txt`)

In [2]:
obs_columns = ['sample', 'cellline', 'treatment']
pdata_pd = pAnnData.import_proteomeDiscoverer(prot_file='pd_prot.txt', pep_file='pd_pep.txt', obs_columns=obs_columns)

🧭 [USER] Importing data of type [pd]
--------------------------
Starting import [Proteome Discoverer]

Source file: pd_prot.txt / pd_pep.txt
Number of files: 12
Proteins: 4988
Peptides: 30920

ℹ️ Using 'Modifications' for modification annotation.

ℹ️ 25 proteins with missing gene names.
     🌐 [API] Querying UniProt for batch 1/1 (25 proteins) [fields: accession, gene_primary]
     ✅ Retrieved UniProt metadata for 25 entries.
     ✅ [OK] Recovered 24 gene name(s) from UniProt. Genes found:
         TUFM, HDLBP, AMPD2, MYG1, HSD17B11, PCM1, NEFH, OXA1L, TRMT5, SLC4A1AP...
     ⚠️ [WARN] 1 gene name(s) still missing. Assigned as 'UNKNOWN_<accession>' for:
         Q6ZSR9
     💡 Tip: You can update these using `pdata.update_identifier_maps({'GENE': 'ACCESSION'}, on='protein', direction='reverse', overwrite=True)`

ℹ️ Removed 64 empty proteins (all-NaN or all-zero). Proteins: P49448, P53675, Q8IZP2, Q15173, O43402, P40123, O95954, O75382, Q9NYQ6, Q14BN4, ...
     ℹ️ [INFO] Using global q-v

## 1. Fresh import → save → load

In [3]:
path = pdata_pd.save(out / 'pd_fresh')   # no extension → '.pdata' is appended
size(path)

🧭 [USER] Saving pAnnData → pd_fresh.pdata
     ℹ️ .prot: 12 samples × 4924 features, 2 layers
     ℹ️ .pep: 12 samples × 30856 features, 2 layers
     ℹ️ .stats: 0 entries, .history: 1 entries
     ✅ [OK] Saved to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\pd_fresh.pdata
 Load with: pdata = pAnnData.load('saveload_out\\pd_fresh.pdata')
saveload_out\pd_fresh.pdata: 10.4 MB


In [4]:
pd_loaded = pAnnData.load(out / 'pd_fresh')   # extension optional here too
pd_loaded

🧭 [USER] Loading pAnnData ← pd_fresh.pdata
     ✅ [OK] Load complete. Use `print(pdata)` to view the object.


pAnnData object

Summary:
  Groups: sample, cellline, treatment, mbr_count, unique_pep2_protein_count
  Avg proteins/sample: 4720.3
  Avg protein quant: 0.96
  Samples < 50% quant: 0

Protein (shape: 12 files × 4924 proteins)
  obs:    sample, cellline, treatment, protein_quant, protein_count...
  var:    Proteins Unique Sequence ID, Checked, Protein FDR Confidence: Combined, Master, Accession...
  obsm:   —
  layers: X_raw, X_mbr

Peptide (shape: 12 files × 30856 peptides)
  obs:    sample, cellline, treatment, peptide_quant, peptide_count...
  var:    Peptide Groups Peptide Group ID, Checked, Confidence, Annotated Sequence, Modifications...
  obsm:   —
  layers: X_raw, X_mbr

RS (shape: 4924 proteins × 30856 peptides)
  Avg peptides/protein: 6.69
  Avg unique peptides : 5.93
  Proteins with ≥2 unique peptides: 76.4%

STRING Enrichment:
  —

In [5]:
same(pdata_pd, pd_loaded)
pd_loaded.metadata

✅ identical (apart from the last 1 history line)


{'source': 'proteomeDiscoverer',
 'prot_file': 'pd_prot.txt',
 'pep_file': 'pd_pep.txt',
 'imported_at': '2026-11-01T00:16:47.125591'}

## 2. Path rules: refuse to overwrite, `overwrite=True`

In [6]:
try:
    pdata_pd.save(out / 'pd_fresh')
except FileExistsError as e:
    print('raised as expected:', e)

pdata_pd.save(out / 'pd_fresh', overwrite=True)

raised as expected:      ❌ [ERROR] saveload_out\pd_fresh.pdata already exists. Pass overwrite=True to replace it.
🧭 [USER] Saving pAnnData → pd_fresh.pdata
     ℹ️ .prot: 12 samples × 4924 features, 2 layers
     ℹ️ .pep: 12 samples × 30856 features, 2 layers
     ℹ️ .stats: 0 entries, .history: 1 entries
     ✅ [OK] Saved to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\pd_fresh.pdata
 Load with: pdata = pAnnData.load('saveload_out\\pd_fresh.pdata')


WindowsPath('saveload_out/pd_fresh.pdata')

## 3. Analysis chain on PD

In [7]:
pdata_pd = pdata_pd.filter_prot(condition='unique_peptides > 1')  # return_copy=False crashes with peptides (todo bug-10)
# annotate after filtering: found metrics aren't subset by later filters (todo bug-11)
pdata_pd.annotate_found(classes='cellline', on='protein')
pdata_pd.annotate_found(classes='cellline', on='peptide')
pdata_pd.normalize(method='median')
pdata_pd.impute(classes=['cellline', 'treatment'], method='mean')
pdata_pd.pca(on='protein')
pdata_pd.neighbor(on='protein')
pdata_pd.umap(on='protein')
pdata_pd.leiden(on='protein', resolution=0.5)

🧭 [USER] Filtering proteins [condition]:
    Returning a copy of protein data with the following filters applied:
     🔸 condition — unique_peptides > 1
    → Proteins kept: 3762
    → Peptides kept (linked): 29474
     ✅ [OK] RS matrix filtered: 3762 proteins, 29474 peptides retained.
🧭 [USER] Annotated 'found in' features by group: cellline using threshold 0.0.
🧭 [USER] Annotated 'found in' features by group: cellline using threshold 0.0.
🧭 [USER] Global normalization using 'median'. Layer will be saved as 'X_norm_median'.
     ✅ Normalized all 12 samples.
     ℹ️ Set protein data to layer X_norm_median.
🧭 [USER] Group-wise imputation using 'mean' on class(es): ['cellline', 'treatment']. Layer saved as 'X_impute_mean'. 
     ✅ 244 values imputed total.
     ℹ️ Group-wise summary:
     Group                | Values Imputed | Skipped Features | Samples Imputed (Partial,Fully)/Total
     ------------------------------------------------------------------------------------------------
   

In [8]:
values = [{'cellline': 'AS', 'treatment': 'kd'}, {'cellline': 'AS', 'treatment': 'sc'}]
de_mean = pdata_pd.de(values)
de_peppair = pdata_pd.de(values, fold_change_mode='pep_pairwise_median')
pdata_pd.de([{'cellline': 'BE'}, {'cellline': 'AS'}])
[k for k in pdata_pd.stats if ' vs ' in k]

     ⚠️ [WARN] Layer 'X' appears to contain non-log intensities (median=3.93e+06). Statistical tests assume approximately normal distributions — consider log_transform() before DE analysis.
🧭 [USER] Running differential expression [protein]
   🔸 Comparing groups: [{'cellline': 'AS', 'treatment': 'kd'}] vs [{'cellline': 'AS', 'treatment': 'sc'}]
   🔸 Group sizes: 3 vs 3 samples
   🔸 Layer: X (non-log)
   🔸 Method: ttest | Fold Change: mean
   🔸 P-value threshold: 0.05 | Log2FC threshold: 1.0
     ℹ️ [INFO] 37 proteins were not comparable (zero or NaN mean in one group).
     ✅ DE complete. Results stored in:
       • .stats["[{'cellline': 'AS', 'treatment': 'kd'}] vs [{'cellline': 'AS', 'treatment': 'sc'}]"]
       • Columns: log2fc, p_value, significance, etc.
       • Upregulated: 22 | Downregulated: 21 | Not significant: 3682 | Not comparable: 37
     ⚠️ [WARN] Layer 'X' appears to contain non-log intensities (median=3.93e+06). Statistical tests assume approximately normal distributi

["[{'cellline': 'AS', 'treatment': 'kd'}] vs [{'cellline': 'AS', 'treatment': 'sc'}]",
 "[{'cellline': 'BE'}] vs [{'cellline': 'AS'}]"]

In [9]:
pdata_pd.pairwise_correlation(classes='cellline')
pdata_pd.cv(classes='cellline')
pdata_pd.rank(classes='cellline')
g = scplt.plot_clustered_heatmap(pdata_pd, classes=['cellline', 'treatment'],
                                 proteins=pdata_pd.prot.var['Genes'].dropna().astype(str).unique()[:30].tolist())
plt.close('all')

     ℹ️ [INFO] Layer 'X' appears to contain non-log intensities (median=3.93e+06). Pairwise correlations on non-log data are dominated by highly abundant proteins. Consider normalize() then log_transform(), then rerun with the resulting log layer.
🧭 [USER] Computing pairwise correlation [protein] using layer: X
     🔸 classes: cellline
     🔸 cellline: AS, BE
     🔸 method: pearson
     🔸 order (N=2): AS | BE
     🔸 Features: 3733 used / 3762 total (29 dropped — NaN in ≥1 group mean)
     ✅ Pairwise correlation complete. Results stored in:
       • .prot.uns['pairwise_corr']
       • Group matrix: (2 × 2)
     ℹ️ [INFO] display_scale='zscore' (auto: default (explicit protein list / no DE stats_key)).
     ℹ️ [INFO] 'PKM' resolved to multiple accessions: ['P14618-2', 'P14618']
     ⚠️ [WARN] Layer 'X' appears non-log (median=3.9e+06). Applying log2(x + 1.0) in memory for this heatmap only. To persist: `pdata.log_transform(layer='X', base=2)` then `layer='X_log2'`.


In [10]:
# needs internet (STRING)
genelist = ['P55072', 'NPLOC4', 'UFD1', 'STX5A', 'NSFL1C', 'UBXN2A', 'UBXN4', 'UBE4B', 'YOD1', 'WASHC5', 'PLAA', 'UBXN10']
pdata_pd.enrichment_functional(genes=genelist, from_de=False)
pdata_pd.enrichment_ppi(genes=genelist)
pdata_pd.list_enrichments()

🧭 [USER] Running STRING enrichment [user-supplied]
     ℹ️ Found 0 cached STRING IDs. 8 need lookup.
     🌐 [API] UniProt mapped: 7 / 8
     🌐 [API] STRING mapped: 0 / 1 (missing after UniProt)
          ℹ️ Cached 7 STRING ID mappings.
          ⚠️ 1 identifier(s) still unmapped: Q92890-1
          Tip: look these up on STRING, then `pdata.update_string_mapping({'ACCESSION': '9606.ENSP...'})`.
   🔸 Input genes: 12 → Resolved STRING IDs: 7
   🔸 Species: 9606 | Background: None
     ⚠️ [WARN] Some accessions unresolved: STX5A, UBXN2A, YOD1, UBXN10
     ✅ [OK] Enrichment complete (1.76s)
   • Access result: pdata.stats['functional']["UserSearch1"]["result"]
   • Plot command : pdata.plot_enrichment_svg("UserSearch1")
   • View online  : https://string-db.org/cgi/network?identifiers=9606.ENSP00000351777%0d9606.ENSP00000331487%0d9606.ENSP00000418529%0d9606.ENSP00000272638%0d9606.ENSP00000343001%0d9606.ENSP00000318016%0d9606.ENSP00000380460&caller_identity=scpviz&species=9606&show_query_node

In [11]:
pdata_pd.update_identifier_maps({pdata_pd.prot.var_names[0]: 'NEWGENE'}, on='protein', direction='reverse', overwrite=True)
pdata_pd.print_history()

[update_identifier_maps] Updated 'protein' (reverse): 0 added, 1 overwritten, 0 skipped.
🔁 Updated `.prot.var['Genes']` for 1 entries from custom mapping. (View details in `pdata.metadata['identifier_map_history']`)
-------------------------------
History:
-------------------------------
1: [2026-11-01 00:16:47] Imported Proteome Discoverer data using source file(s): pd_prot.txt, pd_pep.txt. Queried UniProt API to fill missing gene names in .prot.var['Genes'].
2: [2026-11-01 00:18:58] Pushed summary values back to obs. Skipped for prot: peptide_quant, peptide_count, peptide_abundance_sum, unique_pep2_protein_count. Skipped for pep: protein_quant, protein_count, protein_abundance_sum, unique_pep2_protein_count. 
3: [2026-11-01 00:18:58] 🧭 [USER] Filtering proteins [condition]:
    Returning a copy of protein data with the following filters applied:
     🔸 condition — unique_peptides > 1
    → Proteins kept: 3762
    → Peptides kept (linked): 29474
     ✅ [OK] RS matrix filtered: 3762 pr

## 4. Save → load after the chain

In [12]:
path = pdata_pd.save(out / 'pd_chain')
size(path)
pd_loaded = pAnnData.load(path)
same(pdata_pd, pd_loaded)

🧭 [USER] Saving pAnnData → pd_chain.pdata
     ℹ️ .prot: 12 samples × 3762 features, 4 layers
     ℹ️ .pep: 12 samples × 29474 features, 2 layers
     ℹ️ .stats: 5 entries, .history: 23 entries
     ✅ [OK] Saved to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\pd_chain.pdata
 Load with: pdata = pAnnData.load('saveload_out\\pd_chain.pdata')
saveload_out\pd_chain.pdata: 13.0 MB
🧭 [USER] Loading pAnnData ← pd_chain.pdata
     ✅ [OK] Load complete. Use `print(pdata)` to view the object.
✅ identical (apart from the last 1 history line)


In [13]:
pd_loaded.print_history()

-------------------------------
History:
-------------------------------
1: [2026-11-01 00:16:47] Imported Proteome Discoverer data using source file(s): pd_prot.txt, pd_pep.txt. Queried UniProt API to fill missing gene names in .prot.var['Genes'].
2: [2026-11-01 00:18:58] Pushed summary values back to obs. Skipped for prot: peptide_quant, peptide_count, peptide_abundance_sum, unique_pep2_protein_count. Skipped for pep: protein_quant, protein_count, protein_abundance_sum, unique_pep2_protein_count. 
3: [2026-11-01 00:18:58] 🧭 [USER] Filtering proteins [condition]:
    Returning a copy of protein data with the following filters applied:
     🔸 condition — unique_peptides > 1
    → Proteins kept: 3762
    → Peptides kept (linked): 29474
     ✅ [OK] RS matrix filtered: 3762 proteins, 29474 peptides retained.
4: protein: Annotated features 'found in' class combinations cellline using threshold 0.0.
5: peptide: Annotated features 'found in' class combinations cellline using threshold 0.0.
6

In [14]:
pd_loaded.list_enrichments()
pd_loaded.stats['functional']['UserSearch1']['result'].head()

🧭 [USER] Listing STRING enrichment status

     ℹ️ Available DE comparisons (not yet enriched):
        - AS_kd vs AS_sc
        - BE vs AS

  🔹 To run enrichment:
      pdata.enrichment_functional(from_de=True, de_key="...")

✅ Completed STRING enrichment results:
  - UserSearch1 (User)
    • Table: pdata.stats['functional']["UserSearch1"]['result']
    • Plot : pdata.plot_enrichment_svg("UserSearch1")
    • Link  : https://string-db.org/cgi/network?identifiers=9606.ENSP00000351777%0d9606.ENSP00000331487%0d9606.ENSP00000418529%0d9606.ENSP00000272638%0d9606.ENSP00000343001%0d9606.ENSP00000318016%0d9606.ENSP00000380460&caller_identity=scpviz&species=9606&show_query_node_labels=1

✅ Completed STRING enrichment results:
  - UserPPI1 (User)
    • Table: pdata.stats['ppi']['UserPPI1']['result']


,category,term,number_of_genes,number_of_genes_in_background,ncbiTaxonId,inputGenes,preferredNames,p_value,fdr,description
0,COMPARTMENTS,GOCC:1990730,2,2,9606,"[9606.ENSP00000351777, 9606.ENSP00000418529]","[VCP, NSFL1C]",1.080000e-07,0.000190,VCP-NSFL1C complex
1,COMPARTMENTS,GOCC:0034098,2,8,9606,"[9606.ENSP00000331487, 9606.ENSP00000351777]","[NPLOC4, VCP]",3.030000e-06,0.002600,VCP-NPL4-UFD1 AAA ATPase complex
2,Process,GO:0010498,6,462,9606,"[9606.ENSP00000272638, 9606.ENSP00000331487, 9...","[UBXN4, NPLOC4, UBE4B, VCP, PLAA, NSFL1C]",1.110000e-09,0.000014,Proteasomal protein catabolic process
3,Process,GO:0036503,4,93,9606,"[9606.ENSP00000272638, 9606.ENSP00000331487, 9...","[UBXN4, NPLOC4, UBE4B, VCP]",1.610000e-08,0.000066,ERAD pathway
4,Process,GO:0043161,5,376,9606,"[9606.ENSP00000331487, 9606.ENSP00000343001, 9...","[NPLOC4, UBE4B, VCP, PLAA, NSFL1C]",5.020000e-08,0.000120,Proteasome-mediated ubiquitin-dependent protei...


## 5. Keep working on the loaded object

In [15]:
key = "[{'cellline': 'BE'}] vs [{'cellline': 'AS'}]"
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
scplt.plot_volcano(axes[0], pdata=pdata_pd, stats_key=key); axes[0].set_title('original')
scplt.plot_volcano(axes[1], pdata=pd_loaded, stats_key=key); axes[1].set_title('loaded')
plt.tight_layout()

3 [-0.05801832  0.64870355]
4 [ 0.86224482 -0.34305853]


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
scplt.plot_umap(axes[0], pdata_pd, color='leiden'); axes[0].set_title('original')
scplt.plot_umap(axes[1], pd_loaded, color='leiden'); axes[1].set_title('loaded')
plt.tight_layout()

     ⚠️ [WARN] UMAP already exists in protein data, using existing UMAP. Run with `force=True` to recompute.
     ⚠️ [WARN] UMAP already exists in protein data, using existing UMAP. Run with `force=True` to recompute.


In [17]:
# fresh DE on the loaded object matches the DE stored before saving
fresh = pd_loaded.de([{'cellline': 'BE'}, {'cellline': 'AS'}])
numeric = ['log2fc', 'p_value', 'test_statistic']
pd.testing.assert_frame_equal(fresh[numeric], pdata_pd.stats[key][numeric])
print('✅ fresh de() matches')

     ⚠️ [WARN] Layer 'X' appears to contain non-log intensities (median=3.94e+06). Statistical tests assume approximately normal distributions — consider log_transform() before DE analysis.
🧭 [USER] Running differential expression [protein]
   🔸 Comparing groups: [{'cellline': 'BE'}] vs [{'cellline': 'AS'}]
   🔸 Group sizes: 6 vs 6 samples
   🔸 Layer: X (non-log)
   🔸 Method: ttest | Fold Change: mean
   🔸 P-value threshold: 0.05 | Log2FC threshold: 1.0
     ℹ️ [INFO] 29 proteins were not comparable (zero or NaN mean in one group).
     ✅ DE complete. Results stored in:
       • .stats["[{'cellline': 'BE'}] vs [{'cellline': 'AS'}]"]
       • Columns: log2fc, p_value, significance, etc.
       • Upregulated: 427 | Downregulated: 448 | Not significant: 2858 | Not comparable: 29
✅ fresh de() matches


In [18]:
# filter using the saved 'found in' metrics; peptide-level DE needs the RS + metadata['source']
filtered = pd_loaded.filter_prot_found(group=['AS', 'BE'], min_count=3)
pd_loaded.de(values, fold_change_mode='pep_pairwise_median').head()

🧭 [USER] Filtering proteins [Found|Group-mode|ALL]:
    Returning a copy of protein data based on detection thresholds:
     🔸 Groups requested: ['AS', 'BE']
     🔸 Minimum count: 3
     🔸 Logic: all (protein must be detected in all group(s))
    → Proteins kept: 3697, Proteins dropped: 65
     ✅ [OK] RS matrix filtered: 3697 proteins, 29255 peptides retained.

     ⚠️ [WARN] Layer 'X' appears to contain non-log intensities (median=3.93e+06). Statistical tests assume approximately normal distributions — consider log_transform() before DE analysis.
🧭 [USER] Running differential expression [protein]
   🔸 Comparing groups: [{'cellline': 'AS', 'treatment': 'kd'}] vs [{'cellline': 'AS', 'treatment': 'sc'}]
   🔸 Group sizes: 3 vs 3 samples
   🔸 Layer: X (non-log)
   🔸 Method: ttest | Fold Change: pep_pairwise_median
   🔸 P-value threshold: 0.05 | Log2FC threshold: 1.0
     ℹ️ [INFO] 5070 peptides were not comparable (zero or NaN mean in one group).
     ℹ️ [INFO] 130 proteins were not compar

,Genes,"[{'cellline': 'AS', 'treatment': 'kd'}]","[{'cellline': 'AS', 'treatment': 'sc'}]",log2fc,p_value,test_statistic,-log10(p_value),significance_score,significance
P04083,ANXA1,5.435494e+07,3.108660e+07,1.034373,0.000248,12.341252,3.606060,3.730010,upregulated
P26232,CTNNA2,1.738243e+06,6.487644e+05,1.400934,0.012308,4.334290,1.909810,2.675517,upregulated
P13995,MTHFD2,2.374106e+06,1.505189e+06,1.059732,0.005346,5.494901,2.271972,2.407681,upregulated
Q15942,ZYX,1.814948e+07,1.264367e+07,1.055565,0.002880,6.506331,2.540670,2.681843,upregulated
Q16666,IFI16,1.563568e+07,7.399209e+06,1.018673,0.000595,9.853215,3.225409,3.285637,upregulated


In [19]:
# loaded object → copy → save → load still round-trips
pd_loaded2 = pAnnData.load(filtered.save(out / 'pd_filtered', overwrite=True))
same(filtered, pd_loaded2)

🧭 [USER] Saving pAnnData → pd_filtered.pdata
     ℹ️ .prot: 12 samples × 3697 features, 4 layers
     ℹ️ .pep: 12 samples × 29255 features, 2 layers
     ℹ️ .stats: 5 entries, .history: 27 entries
     ✅ [OK] Saved to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\pd_filtered.pdata
 Load with: pdata = pAnnData.load('saveload_out\\pd_filtered.pdata')
🧭 [USER] Loading pAnnData ← pd_filtered.pdata
     ✅ [OK] Load complete. Use `print(pdata)` to view the object.
✅ identical (apart from the last 1 history line)


## 6. `export_h5ad` (for scanpy users; drops the other modality, rs, summary, history)

In [20]:
h5 = pdata_pd.export_h5ad(out / 'pd_prot', on='protein', overwrite=True)
h5_pep = pdata_pd.export_h5ad(out / 'pd_pep', on='peptide', overwrite=True)

🧭 [USER] Exporting .prot → pd_prot.h5ad
     ⚠️ [WARN] export_h5ad keeps only .prot; not exported: .pep, .rs, .summary, .history. Use pdata.save() to keep everything.
     ⚠️ [WARN] 5 value(s) converted to h5ad-compatible forms:
   - uns['found_metrics_protein']: MultiIndex columns joined with '|'
   - uns['scpviz']['stats']['prot_X_clustered_heatmap']['fingerprint']['sample_set']: tuple → list
   - uns['scpviz']['stats']['functional']['UserSearch1']['result']['inputGenes']: mixed object column → str
   - uns['scpviz']['stats']['functional']['UserSearch1']['result']['preferredNames']: mixed object column → str
   - uns['scpviz']['metadata']['identifier_map_history']: list of containers → dict keyed by position
     ✅ [OK] Exported to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\pd_prot.h5ad
🧭 [USER] Exporting .pep → pd_pep.h5ad
     ⚠️ [WARN] export_h5ad keeps only .pep; not exported: .prot, .rs, .summary, .history. Use pdata.

In [21]:
adata = sc.read_h5ad(h5)
print(adata)
print(adata.uns['scpviz'].keys())
print(adata.uns['scpviz']['metadata'])

AnnData object with n_obs × n_vars = 12 × 3762
    obs: 'sample', 'cellline', 'treatment', 'protein_quant', 'protein_count', 'protein_abundance_sum', 'mbr_count', 'high_count', 'leiden'
    var: 'Proteins Unique Sequence ID', 'Checked', 'Protein FDR Confidence: Combined', 'Master', 'Accession', 'Description', 'Global_Q_value', 'Sum PEP Score', 'Coverage [%]', '# Peptides', '# PSMs', 'unique_peptides', '# AAs', 'MW [kDa]', 'calc. pI', 'Score Sequest HT: Sequest HT', '# Peptides (by Search Engine): Sequest HT', 'Biological Process', 'Cellular Component', 'Molecular Function', 'Pfam IDs', 'Entrez Gene ID', 'Genes', 'Gene ID', 'Ensembl Gene ID', 'Reactome Pathways', 'WikiPathways', '# Protein Pathway Groups', '# Razor Peptides', 'Found in Sample: [S4] F4: Sample, AS, kd', 'Found in Sample: [S8] F8: Sample, AS, kd', 'Found in Sample: [S12] F12: Sample, AS, kd', 'Found in Sample: [S2] F2: Sample, AS, sc', 'Found in Sample: [S6] F6: Sample, AS, sc', 'Found in Sample: [S10] F10: Sample, AS, sc

# DIA-NN import (`tests/test_diann.parquet`)

In [22]:
obs_columns = ['name', 'date', 'MS', 'acquisition', 'FAIMS', 'column', 'gradient', 'amt', 'region', 'replicate']
pdata_diann = pAnnData.import_diann(str(Path('..') / 'tests' / 'test_diann.parquet'), obs_columns=obs_columns)
pdata_diann.prot.obs[['gradient', 'amt', 'replicate']].value_counts()

🧭 [USER] Importing data of type [diann]
--------------------------
Starting import [DIA-NN]

Source file: ..\tests\test_diann.parquet
Number of files: 12
Proteins: 2251
Peptides: 7688

     ℹ️ [INFO] Using sample-specific q-values for 'prot' significance annotation.
     ℹ️ [INFO] Using sample-specific q-values for 'pep' significance annotation.

ℹ️ RS matrix: (2251, 7688) (proteins × peptides), sparsity: 99.95%
   - Proteins with ≥2 *unique* linked peptides: 1217/2251
   - Peptides linked to ≥2 proteins: 158/7688
   - Mean peptides per protein: 3.53
   - Mean proteins per peptide: 1.03
     ✅ [OK] pAnnData object is valid.
     ✅ [OK] Import complete. Use `print(pdata)` to view the object.
--------------------------


gradient      amt  replicate
24min         sc   01           1
                   02           1
                   03           1
30min         sc   01           1
                   02           1
                   03           1
60min         sc   01           1
                   02           1
                   03           1
60minOLDgrad  sc   01           1
                   02           1
                   03           1
Name: count, dtype: int64

## 1. Fresh import → save → load

In [23]:
path = pdata_diann.save(out / 'diann_fresh', overwrite=True)
size(path)
diann_loaded = pAnnData.load(path)
same(pdata_diann, diann_loaded)
diann_loaded

🧭 [USER] Saving pAnnData → diann_fresh.pdata
     ℹ️ .prot: 12 samples × 2251 features, 2 layers
     ℹ️ .pep: 12 samples × 7688 features, 3 layers
     ℹ️ .stats: 0 entries, .history: 1 entries
     ✅ [OK] Saved to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\diann_fresh.pdata
 Load with: pdata = pAnnData.load('saveload_out\\diann_fresh.pdata')
saveload_out\diann_fresh.pdata: 1.6 MB
🧭 [USER] Loading pAnnData ← diann_fresh.pdata
     ✅ [OK] Load complete. Use `print(pdata)` to view the object.
✅ identical (apart from the last 1 history line)


pAnnData object

Summary:
  Groups: name, date, MS, acquisition, FAIMS, column, gradient, amt, region, replicate
  Avg proteins/sample: 968.2
  Avg protein quant: 0.43
  Samples < 50% quant: 8

Protein (shape: 12 files × 2251 proteins)
  obs:    name, date, MS, acquisition, FAIMS...
  var:    Genes, Global_Q_value, peptides_per_protein, unique_peptides, Found In: MP_20250219_OA_DIA_FAIMS_TS25_30min_sc_LCM-Cortex_01...
  obsm:   —
  layers: X_raw, X_qval

Peptide (shape: 12 files × 7688 peptides)
  obs:    name, date, MS, acquisition, FAIMS...
  var:    Genes, Protein.Group, Precursor.Charge, Modified.Sequence, Stripped.Sequence...
  obsm:   —
  layers: X_raw, X_qval, X_precursor

RS (shape: 2251 proteins × 7688 peptides)
  Avg peptides/protein: 3.53
  Avg unique peptides : 3.35
  Proteins with ≥2 unique peptides: 54.1%

STRING Enrichment:
  —

## 2. Analysis chain on DIA-NN

In [24]:
pdata_diann.annotate_significant(classes=['amt'], fdr_threshold=0.05, on='protein')
pdata_diann.annotate_found(classes='gradient', on='protein')
pdata_diann.normalize(method='median')
pdata_diann.impute(method='mean')
pdata_diann.pca(on='protein')
pdata_diann.de([{'gradient': '24min'}, {'gradient': '60min'}])
pdata_diann.pairwise_correlation(classes='gradient')

🧭 [USER] Annotated significant features by group: ['amt'] using FDR threshold 0.05.
🧭 [USER] Annotated 'found in' features by group: gradient using threshold 0.0.
🧭 [USER] Global normalization using 'median'. Layer will be saved as 'X_norm_median'.
     ⚠️ [WARN] 8 sample(s) have >50% missing values.
     Try running `.impute()` before normalization. Suggest to use the flag `use_nonmissing=True` to normalize using only consistently observed proteins.
     ➡️ Use `force=True` to proceed anyway.
🧭 [USER] Global imputation using 'mean'. Layer saved as 'X_impute_mean'. 
     ✅ 15371 values imputed.
     ℹ️ 12 samples fully imputed, 0 samples partially imputed, 0 skipped feature(s) with all missing values.
     ℹ️ Set protein data to layer X_impute_mean.
     ℹ️ [INFO] Layer 'X' appears to contain non-log intensities (median=2.71e+04). PCA will proceed with z-scoring (standard practice). Optionally run pdata.log_transform() first for better low-abundance separation.
🧭 [USER] Performing PCA 

In [25]:
# mixed model (stores per-contrast frames with df.attrs + a collection)
mixed = pdata_diann.mixed_de(group_col='gradient', contrast=('24min', '60min'), donor_col='replicate', store=True,
                             observation_level='pseudobulk', min_detected_fraction=0.0, min_cells_detected=1)
[k for k in pdata_diann.stats]

     ⚠️ [WARN] Layer 'X' appears non-log (median=2.7e+04). Applying log2(x + 1.0) in memory for this mixed_de run only. To persist: `pdata.log_transform(layer='X', base=2)` then `layer='X_log2'`.
🧭 [USER] Running mixed differential expression [protein]
   🔸 Comparing: [{'gradient': '24min'}] vs [{'gradient': '60min'}] | 3 vs 3 samples (before filter) | simple / specified
   🔸 Formula: Pseudobulk: donor-averaged by replicate × gradient
   🔸 Donor blocking: replicate (pseudobulk averaging) | Fixed covariates: (none)
   🔸 Layer: 'X'→'X_impute_mean' (auto log2, pseudocount=1.0) | Method: auto | Level: pseudobulk
   🔸 Adj p-value threshold: 0.05 | Log2FC threshold: 1.0
     ℹ️ [INFO] Mixed DE design diagnostics
       🔸 Donors: 3 total | 3/3 paired (0 24min-only, 0 60min-only) | 100% cells from paired
       🔸 Features: 2250/2251 (1 excluded) | Group sizes after filtering: 3 vs 3 samples
       🔸 Per-feature testing (auto): pseudobulk (paired)=1440, failed=811
     ✅ Mixed DE complete. Resu

["[{'gradient': '24min'}] vs [{'gradient': '60min'}]",
 "mixed: [{'gradient': '24min'}] vs [{'gradient': '60min'}] | donor=replicate"]

## 3. Save → load after the chain

In [26]:
path = pdata_diann.save(out / 'diann_chain', overwrite=True)
size(path)
diann_loaded = pAnnData.load(path)
same(pdata_diann, diann_loaded)
diann_loaded.print_history()

🧭 [USER] Saving pAnnData → diann_chain.pdata
     ℹ️ .prot: 12 samples × 2251 features, 3 layers
     ℹ️ .pep: 12 samples × 7688 features, 3 layers
     ℹ️ .stats: 2 entries, .history: 9 entries
     ✅ [OK] Saved to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\diann_chain.pdata
 Load with: pdata = pAnnData.load('saveload_out\\diann_chain.pdata')
saveload_out\diann_chain.pdata: 2.4 MB
🧭 [USER] Loading pAnnData ← diann_chain.pdata
     ✅ [OK] Load complete. Use `print(pdata)` to view the object.
✅ identical (apart from the last 1 history line)
-------------------------------
History:
-------------------------------
1: [2026-11-01 00:24:21] Imported DIA-NN report from ..\tests\test_diann.parquet using PG.MaxLFQ (protein) and Precursor.Normalised (peptide).
2: protein: Annotated significance across classes ['amt'] using FDR threshold 0.05.
3: protein: Annotated features 'found in' class combinations gradient using threshold 0.0.
4

In [27]:
# df.attrs from mixed_de survive
k = [k for k in pdata_diann.stats if 'mixed' in str(k).lower() or isinstance(pdata_diann.stats[k], pd.DataFrame) and pdata_diann.stats[k].attrs]
for key_ in k:
    v = diann_loaded.stats[key_]
    if isinstance(v, pd.DataFrame):
        print(key_, '→ attrs keys:', list(v.attrs))

mixed: [{'gradient': '24min'}] vs [{'gradient': '60min'}] | donor=replicate → attrs keys: ['mixed_de']


In [28]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
scplt.plot_pca(axes[0], pdata_diann, color='gradient'); axes[0].set_title('original')
scplt.plot_pca(axes[1], diann_loaded, color='gradient'); axes[1].set_title('loaded')
plt.tight_layout()

     ℹ️ [INFO] Using existing PCA embedding.
     ℹ️ [INFO] Using existing PCA embedding.


In [29]:
dkey = [k for k in pdata_diann.stats if ' vs ' in k][0]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
scplt.plot_volcano(axes[0], pdata=pdata_diann, stats_key=dkey); axes[0].set_title('original')
scplt.plot_volcano(axes[1], pdata=diann_loaded, stats_key=dkey); axes[1].set_title('loaded')
plt.tight_layout()

Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, can't fit inside the X axis
Some labels are too long, ca

## 4. `export_h5ad`

In [30]:
h5 = pdata_diann.export_h5ad(out / 'diann_prot', on='protein', overwrite=True)
adata = sc.read_h5ad(h5)
print(adata)
print(adata.uns['scpviz'].keys())

🧭 [USER] Exporting .prot → diann_prot.h5ad
     ⚠️ [WARN] export_h5ad keeps only .prot; not exported: .pep, .rs, .summary, .history. Use pdata.save() to keep everything.
     ⚠️ [WARN] 3 value(s) converted to h5ad-compatible forms:
   - uns['significance_metrics_protein']: MultiIndex columns joined with '|'
   - uns['found_metrics_protein']: MultiIndex columns joined with '|'
   - uns['scpviz']['stats']["mixed: [{'gradient': '24min'}] vs [{'gradient': '60min'}] | donor=replicate"]: DataFrame.attrs dropped
     ✅ [OK] Exported to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\diann_prot.h5ad
AnnData object with n_obs × n_vars = 12 × 2251
    obs: 'name', 'date', 'MS', 'acquisition', 'FAIMS', 'column', 'gradient', 'amt', 'region', 'replicate', 'protein_quant', 'protein_count', 'protein_abundance_sum'
    var: 'Genes', 'Global_Q_value', 'peptides_per_protein', 'unique_peptides', 'Found In: MP_20250219_OA_DIA_FAIMS_TS25_30min_sc_LCM

# Hand-built object + `load(source=...)`

A pAnnData built by hand has no import source, so peptide-level functions (`pep_pairwise_median`) can't map peptides.
`load` warns about it; `source=` fills it in, or set `pdata.metadata['source']` yourself.

In [31]:
hand = pAnnData(prot=pdata_pd.prot.copy(), pep=pdata_pd.pep.copy(), rs=pdata_pd.rs.copy())
hand.update_summary(verbose=False)   # constructor leaves .summary empty (todo bug-12)
hand.metadata

{}

In [32]:
hand.save(out / 'hand', overwrite=True)
hand_loaded = pAnnData.load(out / 'hand')           # warns: no source recorded

🧭 [USER] Saving pAnnData → hand.pdata
     ℹ️ .prot: 12 samples × 3762 features, 4 layers
     ℹ️ .pep: 12 samples × 29474 features, 2 layers
     ℹ️ .stats: 0 entries, .history: 0 entries
     ✅ [OK] Saved to G:\Users\srpang\Dropbox (Personal)\4. Caltech Work\Research\Chou-Roukes Group\scviz_git\dev\saveload_out\hand.pdata
 Load with: pdata = pAnnData.load('saveload_out\\hand.pdata')
🧭 [USER] Loading pAnnData ← hand.pdata
     ⚠️ [WARN] No import source recorded; peptide-to-protein mapping will fail. Pass load(..., source=...) or set pdata.metadata['source'] = 'diann' | 'proteomeDiscoverer' | 'maxquant'.
     ✅ [OK] Load complete. Use `print(pdata)` to view the object.


In [33]:
hand_loaded = pAnnData.load(out / 'hand', source='proteomeDiscoverer')
print(hand_loaded.metadata)
hand_loaded.de(values, fold_change_mode='pep_pairwise_median').head()

🧭 [USER] Loading pAnnData ← hand.pdata
     ℹ️ Set .metadata['source'] = 'proteomeDiscoverer'
     ✅ [OK] Load complete. Use `print(pdata)` to view the object.
{'source': 'proteomeDiscoverer'}
     ⚠️ [WARN] Layer 'X' appears to contain non-log intensities (median=3.93e+06). Statistical tests assume approximately normal distributions — consider log_transform() before DE analysis.
🧭 [USER] Running differential expression [protein]
   🔸 Comparing groups: [{'cellline': 'AS', 'treatment': 'kd'}] vs [{'cellline': 'AS', 'treatment': 'sc'}]
   🔸 Group sizes: 3 vs 3 samples
   🔸 Layer: X (non-log)
   🔸 Method: ttest | Fold Change: pep_pairwise_median
   🔸 P-value threshold: 0.05 | Log2FC threshold: 1.0
     ℹ️ [INFO] 5070 peptides were not comparable (zero or NaN mean in one group).
     ℹ️ [INFO] 130 proteins were not comparable (all peptides invalid or missing).
     ✅ DE complete. Results stored in:
       • .stats["[{'cellline': 'AS', 'treatment': 'kd'}] vs [{'cellline': 'AS', 'treatment':

,Genes,"[{'cellline': 'AS', 'treatment': 'kd'}]","[{'cellline': 'AS', 'treatment': 'sc'}]",log2fc,p_value,test_statistic,-log10(p_value),significance_score,significance
P04083,ANXA1,5.435494e+07,3.108660e+07,1.034373,0.000248,12.341252,3.606060,3.730010,upregulated
P26232,CTNNA2,1.738243e+06,6.487644e+05,1.400934,0.012308,4.334290,1.909810,2.675517,upregulated
P13995,MTHFD2,2.374106e+06,1.505189e+06,1.059732,0.005346,5.494901,2.271972,2.407681,upregulated
Q15942,ZYX,1.814948e+07,1.264367e+07,1.055565,0.002880,6.506331,2.540670,2.681843,upregulated
Q16666,IFI16,1.563568e+07,7.399209e+06,1.018673,0.000595,9.853215,3.225409,3.285637,upregulated


# Cleanup

In [ ]:
# shutil.rmtree(out)